In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import numpy as np
import SI_CoRT
import CoRT_builder

import warnings
warnings.filterwarnings("ignore")


## Testing p_value

In [8]:
n_target = 120
n_source = 50
p = 100
K = 5
Ka = 2
h = 30
alpha = 0.05
T = 5
s_len = 10
s_vector = [0.7] * s_len

N = n_target + Ka * n_source
NI = n_target + n_source
# lamda_k_source = 1.5 * np.sqrt(np.log(p)/ N)
# lamda_1_source = 1.5 * np.sqrt(np.log(p)/ NI) 
# lamda_not_source = 1.5 * np.sqrt(np.log(p) / n_target) 

lamda_k_source = 2 * np.sqrt(np.log(p)/ N)
lamda_1_source = 2 * np.sqrt(np.log(p)/ NI) 
lamda_not_source = 2 * np.sqrt(np.log(p) / n_target) 

para_results_storage = []
CoRT_model = CoRT_builder.CoRT(alpha=lamda_not_source)
iteration = 10

cnt1 = 0
cnt2 = 0
cnt3 = 0
cnt4 = 0

for i in range(iteration):
    target_data, source_data = CoRT_model.gen_data(n_target, n_source, p, K, Ka, h, s_vector, s_len, "AR")
    result_dict = SI_CoRT.SI_parametric(n_target, p, K, target_data, source_data, lamda_not_source, lamda_1_source, lamda_k_source, T, s_len)
    if result_dict != None:
        cnt1 += (result_dict['is_signal'] == True)
        cnt2 += (result_dict['is_signal'] == False)
        cnt3 += (result_dict['is_signal'] == True and result_dict['p_value'] <= alpha)
        cnt4 += (result_dict['is_signal'] == False and result_dict['p_value'] <= alpha)
        if i % 10 == 0:
            print(f"is_signal : {result_dict['is_signal']}, p_values[{i}]: {result_dict['p_value']}")
            print(f"FPR: {cnt4 / cnt2}, TPR: {cnt3 / cnt1}")
            print(f"is_not_signal: {int(cnt2), int(cnt4)}")
            print(f"is_signal: {int(cnt1), int(cnt3)}")
            # print("\n")
            print("===========================================================================")
            
        para_results_storage.append(result_dict)

is_signal : True, p_values[0]: 3.621884570037537e-09
FPR: nan, TPR: 1.0
is_not_signal: (0, 0)
is_signal: (1, 1)


In [ ]:
is_signal_cases = [r for r in para_results_storage if r['is_signal']]
not_signal_cases = [r for r in para_results_storage if not r['is_signal']]

false_positives = sum(1 for c in not_signal_cases if c['p_value'] <= alpha)
print(f"len not_signal_cases : {len(not_signal_cases)}")
print(f"false_positives: {false_positives}")
fpr = false_positives / len(not_signal_cases)
print(f"FPR: {fpr:.4f} (Target: {alpha})")
print("\n")
true_positives = sum(1 for r in is_signal_cases if r['p_value'] <= alpha)
print(f"len not_signal_cases : {len(is_signal_cases)}")
print(f"true_positives: {true_positives}")
tpr = true_positives / len(is_signal_cases)
print(f"TPR: {tpr:.4f}")

len not_signal_cases : 265
false_positives: 13
FPR: 0.0491 (Target: 0.05)


len not_signal_cases : 735
true_positives: 203
TPR: 0.2762
